## AURORA Assignee Predictor — Improved Pipeline V4
### Feature Engineering + TF-IDF + MiniLM Embeddings

In [54]:
import os
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
from tqdm import tqdm
import joblib

In [55]:
# -----------------------------------
# 1) CONFIG
# -----------------------------------
MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "assignee_model_v3_aurora.joblib")
LABEL_ENCODER_PATH = os.path.join(MODEL_DIR, "assignee_label_encoder_aurora.joblib")

PROJECTS = {
    "Aurora": {
        "issues":  r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Issues 554.csv",
        "summary": r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Issues summery 568.csv",
        "sprints": r"PM_Kaggle_dataset\AgileScrumSprintVelocityDataSet\Agile Scrum Dataset\Finalized Datasets for Aurora Project\Aurora Sprints 41.csv",
    },
}

ALLOWED_ASSIGNEES = [
    "Maxim Khutornenko",
    "Bill Farner",
    "Kevin Sweeney",
    "Mark Chu-Carroll",
    "Zameer Manji",
    "Joshua Cohen",
    "Brian Wickman",
    "David McLaughlin",
]

# Minimum samples after balancing (mostly irrelevant with oversampling)
MIN_SAMPLES_PER_CLASS = 2

# Optional MiniLM transformer usage
USE_MINILM = False  # set True if you want embeddings

In [56]:
p = PROJECTS["Aurora"]

df_issues  = pd.read_csv(p["issues"])
df_summary = pd.read_csv(p["summary"])
df_sprints = pd.read_csv(p["sprints"])

print("\nIssues Columns:\n", df_issues.columns)
print("\nSummary Columns:\n", df_summary.columns)
print("\nSprints Columns:\n", df_sprints.columns)



Issues Columns:
 Index(['key', 'issueType', 'sprint', 'status', 'summary', 'description',
       'storyPoint', 'priority', 'watchcount', 'fixVersions',
       'affectedVersions', 'assignee', 'creator', 'reporter', 'commentCount',
       'votes', 'issueLinks', 'blockedBy', 'blocks', 'dependedOnBy',
       'dependedOn', 'subtasks'],
      dtype='object')

Summary Columns:
 Index(['projectID', 'sprintId', 'status', 'storyId', 'issueKey', 'summary',
       'priorityId', 'assignee', 'initialStoryPoint', 'currentStoryPoint'],
      dtype='object')

Sprints Columns:
 Index(['sprintId', 'sprintName', 'sprintState', 'sprintStartDate',
       'sprintEndDate', 'sprintCompleteDate', 'total', 'completedIssuesCount',
       'issuesNotCompletedInCurrentSprint', 'puntedIssues',
       'issuesCompletedInAnotherSprint', 'issueKeysAddedDuringSprint',
       'completedIssuesInitialEstimateSum', 'completedIssuesEstimateSum',
       'puntedIssuesInitialEstimateSum', 'puntedIssuesEstimateSum',
       'issue

In [57]:
print(numeric_cols)
df[numeric_cols].dtypes

['kw_scheduler', 'kw_mesos', 'kw_protobuf', 'kw_agent', 'kw_quota', 'kw_api', 'kw_frontend', 'kw_backend', 'kw_refactor', 'kw_migration', 'kw_resource', 'kw_tracking', 'kw_build', 'kw_testing', 'kw_framework', 'kw_executor', 'rpt_Bhuvaneswaran A', 'rpt_Bill Farner', 'rpt_Bjoern Metzdorf', 'rpt_Brian Wickman', 'rpt_Chengwei Yang', 'rpt_Chris Lambert', 'rpt_Dan Norris', 'rpt_David McLaughlin', 'rpt_Dominic Hamon', 'rpt_Florian Pfeiffer', 'rpt_Jake Farrell', 'rpt_Jay Buffington', 'rpt_Joe Smith', 'rpt_Jonathan Boulle', 'rpt_Joshua Cohen', 'rpt_Kevin Sweeney', 'rpt_Mark Chu-Carroll', 'rpt_Maxim Khutornenko', 'rpt_Stephan Erb', 'rpt_Steve Niemitz', 'rpt_Suman Karumuri', 'rpt_Zameer Manji', 'rpt_alexius ludeman', 'summary_len']


kw_scheduler             int64
kw_mesos                 int64
kw_protobuf              int64
kw_agent                 int64
kw_quota                 int64
kw_api                   int64
kw_frontend              int64
kw_backend               int64
kw_refactor              int64
kw_migration             int64
kw_resource              int64
kw_tracking              int64
kw_build                 int64
kw_testing               int64
kw_framework             int64
kw_executor              int64
rpt_Bhuvaneswaran A      int64
rpt_Bill Farner          int64
rpt_Bjoern Metzdorf      int64
rpt_Brian Wickman        int64
rpt_Chengwei Yang        int64
rpt_Chris Lambert        int64
rpt_Dan Norris           int64
rpt_David McLaughlin     int64
rpt_Dominic Hamon        int64
rpt_Florian Pfeiffer     int64
rpt_Jake Farrell         int64
rpt_Jay Buffington       int64
rpt_Joe Smith            int64
rpt_Jonathan Boulle      int64
rpt_Joshua Cohen         int64
rpt_Kevin Sweeney        int64
rpt_Mark

In [58]:
def load_aurora():
    p = PROJECTS["Aurora"]

    df_issues  = pd.read_csv(p["issues"])
    df_summary = pd.read_csv(p["summary"])
    df_sprints = pd.read_csv(p["sprints"])

    print("\n--- Raw Loaded ---")
    print("Issues:", df_issues.shape)
    print("Summary:", df_summary.shape)
    print("Sprints:", df_sprints.shape)

    # Normalize column names (just in case)
    df_issues.columns  = df_issues.columns.str.strip()
    df_summary.columns = df_summary.columns.str.strip()
    df_sprints.columns = df_sprints.columns.str.strip()

    # ------------------------------
    # MERGE 1: Issues ↔ Summary
    # ------------------------------
    if "key" in df_issues.columns and "issueKey" in df_summary.columns:
        df = df_issues.merge(
            df_summary,
            left_on="key",
            right_on="issueKey",
            how="left",
            suffixes=("", "_summary")
        )
    else:
        raise ValueError("❌ Cannot merge Issues and Summary: 'key' or 'issueKey' missing.")

    # ------------------------------
    # MERGE 2: Issues ↔ Sprints
    # ------------------------------
    if "sprint" in df.columns and "sprintId" in df_sprints.columns:
        df = df.merge(
            df_sprints,
            left_on="sprint",
            right_on="sprintId",
            how="left",
            suffixes=("", "_sprint")
        )
    else:
        raise ValueError("❌ Cannot merge Issues and Sprints: 'sprint' or 'sprintId' missing.")

    print("Merged dataset:", df.shape)
    return df


In [59]:
# ============================================================
# 3) FILTER & CLEANING
# ============================================================
df = load_aurora()

df = df[df["assignee"].notna()].copy()
print("\nAfter removing empty assignees:", df.shape)

# Apply whitelist
df = df[df["assignee"].isin(ALLOWED_ASSIGNEES)].copy()
print("After applying whitelist:", df.shape)

df["summary"] = df["summary"].fillna("")
df["description"] = df["description"].fillna("")


--- Raw Loaded ---
Issues: (554, 22)
Summary: (568, 10)
Sprints: (40, 22)
Merged dataset: (893, 54)

After removing empty assignees: (854, 54)
After applying whitelist: (824, 54)


In [60]:
# ============================================================
# 4) FEATURE ENGINEERING
# ============================================================

# --- Keyword Features ---
KEYWORDS = [
    "scheduler","mesos","protobuf","agent","quota","api",
    "frontend","backend","refactor","migration","resource",
    "tracking","build","testing","framework","executor"
]

for kw in KEYWORDS:
    df[f"kw_{kw}"] = df["summary"].str.contains(kw, case=False, na=False).astype(int)

# --- Components ---
if "component" in df.columns:
    df = pd.get_dummies(df, columns=["component"], prefix="comp", dummy_na=True)

# --- Reporter ---
if "reporter" in df.columns:
    df["reporter"] = df["reporter"].fillna("unknown")
    df = pd.get_dummies(df, columns=["reporter"], prefix="rpt")

# --- Created Date ---
if "created" in df.columns:
    df["created"] = pd.to_datetime(df["created"], errors="coerce")
    df["year"] = df["created"].dt.year.fillna(0).astype(int)
    df["month"] = df["created"].dt.month.fillna(0).astype(int)
    df["day"] = df["created"].dt.day.fillna(0).astype(int)
    df["quarter"] = df["created"].dt.quarter.fillna(0).astype(int)

df["summary_len"] = df["summary"].apply(lambda x: len(x.split()))

In [61]:
# ============================================================
# 5) OVERSAMPLING BALANCING
# ============================================================

print("\nBefore balancing:")
print(df["assignee"].value_counts())

max_count = df["assignee"].value_counts().max()
dfs = []

for assignee, count in df["assignee"].value_counts().items():
    df_cls = df[df["assignee"] == assignee]
    if count < max_count:
        # oversample with replacement
        df_over = df_cls.sample(max_count, replace=True, random_state=42)
        dfs.append(df_over)
    else:
        dfs.append(df_cls)

df = pd.concat(dfs).sample(frac=1, random_state=42).reset_index(drop=True)

print("\nAfter oversampling:")
print(df["assignee"].value_counts())


Before balancing:
assignee
Maxim Khutornenko    263
Bill Farner          160
Kevin Sweeney        114
Mark Chu-Carroll      81
Joshua Cohen          68
Zameer Manji          57
Brian Wickman         54
David McLaughlin      27
Name: count, dtype: int64

After oversampling:
assignee
Mark Chu-Carroll     263
Bill Farner          263
Brian Wickman        263
Maxim Khutornenko    263
David McLaughlin     263
Zameer Manji         263
Kevin Sweeney        263
Joshua Cohen         263
Name: count, dtype: int64


In [62]:
# ============================================================
# 6) TEXT VECTORIZATION (TF-IDF)
# ============================================================

tfidf = TfidfVectorizer(
    max_features=500,
    ngram_range=(1,2),
    stop_words="english"
)

X_tfidf = tfidf.fit_transform(df["summary"].values)
print("TF-IDF shape:", X_tfidf.shape)

joblib.dump(tfidf, os.path.join(MODEL_DIR, "tfidf_vectorizer.joblib"))

TF-IDF shape: (2104, 500)


['models\\tfidf_vectorizer.joblib']

In [63]:
# ============================================================
# 7) OPTIONAL MINI-LM EMBEDDINGS
# ============================================================

if USE_MINILM:
    print("\nEncoding MiniLM embeddings...")
    from sentence_transformers import SentenceTransformer
    model_emb = SentenceTransformer("all-MiniLM-L6-v2")

    emb_list = []
    for text in tqdm(df["summary"].values):
        emb_list.append(model_emb.encode(str(text)))

    X_emb = np.vstack(emb_list)
    joblib.dump(model_emb, os.path.join(MODEL_DIR, "minilm_encoder.joblib"))
else:
    X_emb = np.zeros((df.shape[0], 1))

In [64]:
# ============================================================
# 8) FEATURE MATRIX BUILDER
# ============================================================

# 1. Select numeric feature columns
numeric_cols = [
    col for col in df.columns
    if col.startswith("kw_")
    or col.startswith("comp_")
    or col.startswith("rpt_")
    or col in ["year","month","day","quarter","summary_len"]
]

print("\nNumeric feature columns count:", len(numeric_cols))

# 2. Convert ALL numeric columns to safe numeric dtypes
#    - bool → int
#    - object → numeric (coerce to NaN → fill 0)
for col in numeric_cols:
    if df[col].dtype == bool:
        df[col] = df[col].astype(int)

df[numeric_cols] = df[numeric_cols].apply(
    lambda x: pd.to_numeric(x, errors="coerce")
).fillna(0)

# Debug check
print("\nSanitized numeric dtypes:")
print(df[numeric_cols].dtypes)

# 3. Build dense numpy array
X_num_array = df[numeric_cols].astype(float).values

print("\nX_num_array dtype:", X_num_array.dtype)
print("X_num_array shape:", X_num_array.shape)

# 4. Build sparse matrix safely
X_num = csr_matrix(X_num_array)

# 5. Combine TF-IDF + numeric
X_sparse = hstack([X_tfidf, X_num]).tocsr()

print("\nX_sparse shape:", X_sparse.shape)

# 6. Combine with embeddings (or zeros)
if X_emb.ndim == 1:  # if placeholder zeros
    X_emb = X_emb.reshape(-1, 1)

X_final = np.hstack([X_sparse.toarray(), X_emb])

print("\nFINAL X_final shape:", X_final.shape)



Numeric feature columns count: 40

Sanitized numeric dtypes:
kw_scheduler             int64
kw_mesos                 int64
kw_protobuf              int64
kw_agent                 int64
kw_quota                 int64
kw_api                   int64
kw_frontend              int64
kw_backend               int64
kw_refactor              int64
kw_migration             int64
kw_resource              int64
kw_tracking              int64
kw_build                 int64
kw_testing               int64
kw_framework             int64
kw_executor              int64
rpt_Bhuvaneswaran A      int64
rpt_Bill Farner          int64
rpt_Bjoern Metzdorf      int64
rpt_Brian Wickman        int64
rpt_Chengwei Yang        int64
rpt_Chris Lambert        int64
rpt_Dan Norris           int64
rpt_David McLaughlin     int64
rpt_Dominic Hamon        int64
rpt_Florian Pfeiffer     int64
rpt_Jake Farrell         int64
rpt_Jay Buffington       int64
rpt_Joe Smith            int64
rpt_Jonathan Boulle      int64
rpt_Josh

In [65]:
# ============================================================
# 9) LABEL ENCODER
# ============================================================

le = LabelEncoder()
y = le.fit_transform(df["assignee"])

joblib.dump(le, LABEL_ENCODER_PATH)

print("\nLabel Mapping:")
for i, cls in enumerate(le.classes_):
    print(i, cls)


Label Mapping:
0 Bill Farner
1 Brian Wickman
2 David McLaughlin
3 Joshua Cohen
4 Kevin Sweeney
5 Mark Chu-Carroll
6 Maxim Khutornenko
7 Zameer Manji


In [66]:
# ============================================================
# 10) TRAIN/TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X_final, y, test_size=0.20, random_state=42, stratify=y
)


In [67]:
# ============================================================
# 11) TRAIN XGBOOST MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=600,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.7,
    eval_metric="mlogloss",
    tree_method="hist"
)

print("\nTraining model...")
model.fit(X_train, y_train)

joblib.dump(model, MODEL_PATH)



Training model...


['models\\assignee_model_v3_aurora.joblib']

In [68]:
# ============================================================
# 12) EVALUATION
# ============================================================

y_pred = model.predict(X_test)
probs = model.predict_proba(X_test)

top1 = accuracy_score(y_test, y_pred)
top3 = np.mean([y_test[i] in np.argsort(probs[i])[-3:] for i in range(len(y_test))])
top5 = np.mean([y_test[i] in np.argsort(probs[i])[-5:] for i in range(len(y_test))])

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Top-1 Accuracy:", round(top1, 3))
print("Top-3 Accuracy:", round(top3, 3))
print("Top-5 Accuracy:", round(top5, 3))


Classification Report:
                   precision    recall  f1-score   support

      Bill Farner       0.94      0.98      0.96        52
    Brian Wickman       0.98      1.00      0.99        52
 David McLaughlin       1.00      1.00      1.00        53
     Joshua Cohen       0.98      0.98      0.98        53
    Kevin Sweeney       1.00      1.00      1.00        53
 Mark Chu-Carroll       0.98      1.00      0.99        52
Maxim Khutornenko       0.98      0.91      0.94        53
     Zameer Manji       1.00      1.00      1.00        53

         accuracy                           0.98       421
        macro avg       0.98      0.98      0.98       421
     weighted avg       0.98      0.98      0.98       421

Top-1 Accuracy: 0.983
Top-3 Accuracy: 0.995
Top-5 Accuracy: 0.998
